<a href="https://colab.research.google.com/github/emily-escudero/Analitica-Educacion-rural/blob/main/Entrega3EducacionRural.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **Base de datos Deserción Escolar**

Por medio de un API se genera el DataFrame de deserción Escolar por departamento



In [2]:
import pandas as pd
import requests

# Definir la API URL
url = "https://www.datos.gov.co/api/v3/views/ji8i-4anb/query.json"

# solicitud a la API
respuesta_api = requests.get(url)
respuesta_api.raise_for_status()
datos = respuesta_api.json()


df = pd.DataFrame(datos)
display(df.head())

,:id,:version,:created_at,:updated_at,ano,c_digo_departamento,departamento,poblacion_5_16,tasa_matriculacion_5_16,cobertura_neta,...,reprobacion,reprobacion_transicion,reprobacion_primaria,reprobacion_secundaria,reprobacion_media,repitencia,repitencia_transicion,repitencia_primaria,repitencia_secundaria,repitencia_media
0,row-xccu-54yf.9ie2,rv-69kt.ykig.crwc,2024-01-30T16:43:35.793Z,2024-01-30T16:43:35.793Z,2011,5,Antioquia,1288473,94.01,93.85,...,2.06,0.07,94.56,2.54,2.96,4.25,0.07,4.56,5.27,1.68
1,row-755p~4uyd-2k3w,rv-ar5t~pmwz_z3hy,2024-01-30T16:43:35.793Z,2024-01-30T16:43:35.793Z,2011,8,Atlántico,523935,99.32,99.05,...,0.54,0.12,96.49,0.67,0.75,1.82,0.12,1.77,2.18,0.88
2,row-rb29.mx9s.a7ax,rv-atze~ze7f~jiue,2024-01-30T16:43:35.793Z,2024-01-30T16:43:35.793Z,2011,11,"Bogotá, D.C.",1479334,90.7,90.29,...,0,0,94.69,0,0,3.23,0,2.3,5.11,2.57
3,row-gavf~qv4q~dhdy,rv-9idw~dvm2~k7me,2024-01-30T16:43:35.793Z,2024-01-30T16:43:35.793Z,2011,13,Bolívar,496676,91.57,91.4,...,2.1,0.46,95.48,2.75,3.67,4.43,0.46,4.44,5.37,2.28
4,row-zz5g~58au~qymy,rv-68rp.uscp_a52j,2024-01-30T16:43:35.793Z,2024-01-30T16:43:35.793Z,2011,15,Boyacá,300501,86.16,86.11,...,2.73,0.17,96.1,4.31,3.26,2.62,0.17,1.9,4.19,1.55


In [ ]:
# Definir columnas
columnas_desercion = [
    'departamento',
    'ano',
    'desercion',
    'desercion_transicion',
    'desercion_primaria',
    'desercion_secundaria',
    'desercion_media'
]

# Seleccionamos las columnas necesarias
datos_desercion = df[df.columns.intersection(columnas_desercion)].copy()

# Convertir la columna año a formato numerico
datos_desercion['ano'] = pd.to_numeric(datos_desercion['ano'], errors='coerce')

# Filtrar la informacion entre el 2021 y 2024
df_deserciones_2021_2024 = datos_desercion[
    (datos_desercion['ano'] >= 2021) & (datos_desercion['ano'] <= 2024)
].copy()

# Display the first few rows of the filtered DataFrame
display(df_deserciones_2021_2024.head())

,ano,departamento,desercion,desercion_transicion,desercion_primaria,desercion_secundaria,desercion_media
330,2021,Antioquia,4.81,3.59,4.27,5.90,4.13
331,2021,Atlántico,1.67,2.15,1.75,1.68,1.03
332,2021,"Bogotá, D,C,",1.29,1.07,1.08,1.35,1.88
333,2021,Bolívar,3.69,4.13,3.33,4.27,3.13
334,2021,Boyacá,2.97,2.75,2.13,3.63,3.69


### Base de datos Pobreza Monetaria

Se anexa por medio de Github la base de datos de pobreza departamental

In [ ]:
import pandas as pd
import requests

# URL del archivo excel en github
excel_url = "https://github.com/emily-escudero/Analitica-Educacion-rural/raw/main/anex-pobrezadepartamental.xlsx"

# Nombre del archivo
pobreza_departamental = "anex-pobrezadepartamental.xlsx"

# Descargar el archivo excel
print(f"Downloading {excel_url}...")
response = requests.get(excel_url)
response.raise_for_status()

with open(pobreza_departamental, 'wb') as f:
    f.write(response.content)

print(f"Archivo descargado {pobreza_departamental}")

Archivo descargado anex-pobrezadepartamental.xlsx


### Preparación de datos de pobreza monetaria

En este paso se extraen los datos de la hoja Pobreza Monetaria Act.Met. del archivo Excel. Los nombres de los departamentos se encuentran en la columna A, desde la fila 12 hasta la 35, mientras que los años están ubicados en la fila 11, desde las columnas B hasta F. Posteriormente, la información se organiza para facilitar su análisis y filtrado.

In [ ]:
# Leer la fila de la lista_años
tabla_años = pd.read_excel(
    pobreza_departamental,
    sheet_name='Pobreza Monetaria Act.Met.',
    header=None,
    skiprows=10,
    nrows=1,
    usecols='B:F'
)

# Extraer los lista_años a la hoja de escel
lista_años = [int(col) for col in tabla_años.iloc[0].values]

# Leer los datos de pobreza monetaria por departamento
monetaria_df = pd.read_excel(
    pobreza_departamental,
    sheet_name='Pobreza Monetaria Act.Met.',
    header=None,
    skiprows=11,
    nrows=24,
    usecols='A:F'
)

# Asignar nombres a las columnas
monetaria_df.columns = ['departamento'] + lista_años

# convertir la tabla de formato ancho a formato largo
df_monetaria_melted = monetaria_df.melt(
    id_vars=['departamento'],
    var_name='ano',
    value_name='pobreza_monetaria'
)

# Filtrar la lista de años del 2021 al 2024
df_pobreza_monetaria = df_monetaria_melted[
    (df_monetaria_melted['ano'] >= 2021) & (df_monetaria_melted['ano'] <= 2024)
].copy()

print("Pobreza Monetaria (2021-2024):")
display(df_pobreza_monetaria.head())

Pobreza Monetaria (2021-2024):


,departamento,ano,pobreza_monetaria
0,Antioquia,2021,32.8
1,Atlántico,2021,42.1
2,Bogotá D.C.,2021,30.5
3,Bolívar,2021,54.0
4,Boyacá,2021,41.8


### Preparacion de datos de pobreaza extrema

A continuación, se procesan los datos correspondientes a la hoja Pobreza Extrema Act.Met.. En esta hoja, los nombres de los departamentos se encuentran desde la celda A15 hasta A38, mientras que los años están ubicados en la fila 14, desde las columnas B hasta F.

In [ ]:

# Leer la fila donde se encuntran los años
tabla_años_extrema = pd.read_excel(
    pobreza_departamental,
    sheet_name='Pobreza Extrema Act.Met.',
    header=None,
    skiprows=13,
    nrows=1,
    usecols='B:F'
)

# Extraer años de tabla_años_extrema
lista_años_extrema = [int(col) for col in tabla_años_extrema.iloc[0].values]

# Leer los datos de pobreza extrema por departamento
datos_pobreza_extrema = pd.read_excel(
    pobreza_departamental,
    sheet_name='Pobreza Extrema Act.Met.',
    header=None,
    skiprows=14,
    nrows=24,
    usecols='A:F'
)

# Asignar nombres a las columnas
datos_pobreza_extrema.columns = ['departamento'] + lista_años_extrema

# Convertir la tabla de formato ancho a formato largo
df_extrema_melted = datos_pobreza_extrema.melt(
    id_vars=['departamento'],
    var_name='ano',
    value_name='pobreza_extrema'
)

# Filtrar años del 2021 al 2024
df_pobreza_extrema = df_extrema_melted[
    (df_extrema_melted['ano'] >= 2021) & (df_extrema_melted['ano'] <= 2024)
].copy()

# Mostrar primeras filas
print("Pobreza Extrema (2021-2024):")
display(df_pobreza_extrema.head())

Pobreza Extrema (2021-2024):


,departamento,ano,pobreza_extrema
0,Antioquia,2021,9.2
1,Atlántico,2021,12.3
2,Bogotá D.C.,2021,8.4
3,Bolívar,2021,18.8
4,Boyacá,2021,16.4


### Union base de datos

En este paso se integran los datos de pobreza monetaria y pobreza extrema en una sola base de datos, de manera que cada registro contenga la información correspondiente al mismo departamento y año.

In [ ]:
# Unir datos de pobreza monetaria a pobreza extrema
df_pobreza_completos = pd.merge(
    df_pobreza_monetaria,
    df_pobreza_extrema,
    on=['departamento', 'ano'],
    how='inner'
)

# Mostrar base de datos final
print("Final DataFrame with Pobreza Monetaria and Pobreza Extrema (2021-2024):")
display(df_pobreza_completos)

Final DataFrame with Pobreza Monetaria and Pobreza Extrema (2021-2024):


,departamento,ano,pobreza_monetaria,pobreza_extrema
0,Antioquia,2021,32.8,9.2
1,Atlántico,2021,42.1,12.3
2,Bogotá D.C.,2021,30.5,8.4
3,Bolívar,2021,54.0,18.8
4,Boyacá,2021,41.8,16.4
...,...,...,...,...
91,Risaralda,2024,23.8,5.2
92,Santander,2024,27.4,7.7
93,Sucre,2024,57.5,23.7
94,Tolima,2024,35.6,12.2


In [ ]:
import pandas as pd
import requests

# Url del archivo de educacion formal
educacion_formal = "https://github.com/emily-escudero/Analitica-Educacion-rural/raw/main/anex-educacionformal.xlsx"

# Nombre con el que guardo el archivo
ruta_educacion_formal = "anex-educacionformal.xlsx"

# Descargar archivo de excel
print(f"Downloading {educacion_formal}...")
respuesta_descarga = requests.get(educacion_formal)
respuesta_descarga.raise_for_status()

with open(ruta_educacion_formal, 'wb') as f:
    f.write(respuesta_descarga.content)

print(f"Se dercargo el archivo {ruta_educacion_formal}")

Se dercargo el archivo anex-educacionformal.xlsx


### Preparación de Datos de Sedes Educativas

Ahora vamos a cargar los datos de la hoja Sedes_nivel educativo_zona. Según tus indicaciones:
*   Los departamentos están en la columna A desde la fila 19 hasta la 51.
*   El número de sedes urbanas va desde la columna B hasta la F en la fila 17.
*   El número de sedes rurales va desde la columna G hasta la K en la fila 17.

In [ ]:

# 1. leer los nombres de las categorias de educacion
encabezados_educacion = pd.read_excel(
    ruta_educacion_formal,
    sheet_name='Sedes_nivel educativo_zona',
    header=None,
    skiprows=16,
    nrows=1,
    usecols='B:K'
)
nombres_educacion = encabezados_educacion.iloc[0].tolist()

# crear nombres para las columnas  urbanas y rurales
columnas_urbanas = [f'U_{name}' for name in nombres_educacion[:5]] # e.g., U_Preescolar
columnaas_rurales = [f'R_{name}' for name in nombres_educacion[5:]] # e.g., R_Preescolar
columnas_numericas = columnas_urbanas + columnaas_rurales


# 2. Leer los nombres de los departamentos
datos_departamentos = pd.read_excel(
    ruta_educacion_formal,
    sheet_name='Sedes_nivel educativo_zona',
    header=None,
    skiprows=18,
    nrows=33,
    usecols='A'
)
lista_departamentos = datos_departamentos.iloc[:, 0].tolist()

# 3. Leer los datos de sedes educativas urbanas y rurales
datos_sedes = pd.read_excel(
    ruta_educacion_formal,
    sheet_name='Sedes_nivel educativo_zona',
    header=None,
    skiprows=18,
    nrows=33,
    usecols='B:K'
)

# Asignar nombres a las columnas
datos_sedes.columns = columnas_numericas

#  DataFrame
df_instituciones_temp = pd.DataFrame(datos_sedes)
df_instituciones_temp.insert(0, 'departamento', lista_departamentos)

# Calcular el total de sedes educativas
df_instituciones_temp.iloc[:, 1:] = df_instituciones_temp.iloc[:, 1:].fillna(0)



df_instituciones = pd.DataFrame({
    'departamento': df_instituciones_temp['departamento'],
    'total_sedes_urbanas': df_instituciones_temp[columnas_urbanas].sum(axis=1),
    'total_sedes_rurales': df_instituciones_temp[columnaas_rurales].sum(axis=1)
})

df_instituciones['total_sedes'] = df_instituciones['total_sedes_urbanas'] + df_instituciones['total_sedes_rurales']

# Mostrar resultados
print("Total de Instituciones Urbanas y Rurales por Departamento:")
display(df_instituciones.head())
display(df_instituciones.tail())

Total de Instituciones Urbanas y Rurales por Departamento:


,departamento,total_sedes_urbanas,total_sedes_rurales,total_sedes
0,Amazonas,47,161,208
1,Antioquia,4126,9853,13979
2,Arauca,277,846,1123
3,"Archipiélago de San Andrés, Providencia y Sant...",46,35,81
4,Atlántico,3210,271,3481


,departamento,total_sedes_urbanas,total_sedes_rurales,total_sedes
28,Sucre,824,1573,2397
29,Tolima,1501,3488,4989
30,Valle del Cauca,4446,3065,7511
31,Vaupés,25,224,249
32,Vichada,48,462,510


### **Integración de Base de Datos TerriData Vivienda**

En esta sección descargamos y procesamos los indicadores de **Cobertura de energía eléctrica rural** y **Cobertura de Internet Rural (Censo)** a nivel departamental para el período 2021-2024.

In [ ]:
import pandas as pd
import requests

# URL del archivo excel en github
excel_url_vivienda = "https://github.com/emily-escudero/Analitica-Educacion-rural/raw/main/TerriData_vivienda.xlsx"
terridata_vivienda = "TerriData_vivienda.xlsx"

# Descargar el archivo excel
print(f"Downloading {excel_url_vivienda}...")
response = requests.get(excel_url_vivienda)
response.raise_for_status()

with open(terridata_vivienda, 'wb') as f:
    f.write(response.content)

print(f"Archivo descargado exitosamente: {terridata_vivienda}")

Archivo descargado exitosamente: TerriData_vivienda.xlsx


In [ ]:
# Cargar los datos de vivienda (asumiendo Hoja01 como la hoja principal de datos de TerriData)
df_vivienda_raw = pd.read_excel(terridata_vivienda, sheet_name='Hoja01')

# Mostrar las columnas y las primeras filas para entender la estructura exacta del archivo
display(df_vivienda_raw.head())
print("Columnas disponibles:", df_vivienda_raw.columns.tolist())

,Código Departamento,Departamento,Código Entidad,Entidad,Dimensión,Subcategoría,Indicador,Dato Numérico,Dato Cualitativo,Año,Mes,Fuente,Unidad de Medida
0,1,Colombia,1001,Colombia,Vivienda y acceso a servicios públicos,Vivienda y acceso a servicios públicos,Cobertura de energía eléctrica rural,"95,79",NaN,2011,12,Unidad de Planeación Minero-Energética (UPME),Porcentaje (el valor está multiplicado por 100)
1,5,Antioquia,5000,Antioquia,Vivienda y acceso a servicios públicos,Vivienda y acceso a servicios públicos,Cobertura de energía eléctrica rural,"98,05",NaN,2011,12,Unidad de Planeación Minero-Energética (UPME),Porcentaje (el valor está multiplicado por 100)
2,5,Antioquia,5001,Medellín,Vivienda y acceso a servicios públicos,Vivienda y acceso a servicios públicos,Cobertura de energía eléctrica rural,"100,00",NaN,2011,12,Unidad de Planeación Minero-Energética (UPME),Porcentaje (el valor está multiplicado por 100)
3,5,Antioquia,5002,Abejorral,Vivienda y acceso a servicios públicos,Vivienda y acceso a servicios públicos,Cobertura de energía eléctrica rural,"100,00",NaN,2011,12,Unidad de Planeación Minero-Energética (UPME),Porcentaje (el valor está multiplicado por 100)
4,5,Antioquia,5004,Abriaquí,Vivienda y acceso a servicios públicos,Vivienda y acceso a servicios públicos,Cobertura de energía eléctrica rural,"99,48",NaN,2011,12,Unidad de Planeación Minero-Energética (UPME),Porcentaje (el valor está multiplicado por 100)


Columnas disponibles: ['Código Departamento', 'Departamento', 'Código Entidad', 'Entidad', 'Dimensión', 'Subcategoría', 'Indicador', 'Dato Numérico', 'Dato Cualitativo', 'Año', 'Mes', 'Fuente', 'Unidad de Medida']


In [ ]:
import pandas as pd
import unicodedata
import re

# 1. Mapear de forma segura usando los nombres de las columnas correctas
columnas_origen = {
    'Departamento': 'departamento',
    'Entidad': 'entidad',
    'Indicador': 'indicador',
    'Dato Numérico': 'dato_numerico',
    'Año': 'ano'
}

df_vivienda = df_vivienda_raw[list(columnas_origen.keys())].copy()
df_vivienda.columns = [columnas_origen[col] for col in df_vivienda.columns]

# Función para normalizar texto general (remover tildes, minúsculas, etc.)
def normalizar_texto(texto):
    if not isinstance(texto, str):
        return str(texto)
    texto = texto.lower()
    texto = unicodedata.normalize('NFKD', texto).encode('ascii', 'ignore').decode('utf-8')
    texto = re.sub(r'bogota[,\s]*d[.\s]*c[.\s]*', 'bogota dc', texto)
    texto = re.sub(r'[^a-z0-9\s]', '', texto)
    return re.sub(r'\s+', ' ', texto).strip()

# Normalizar la columna indicador
df_vivienda['indicador_clean'] = df_vivienda['indicador'].apply(normalizar_texto)

# Claves deseadas normalizadas sin tildes ni caracteres especiales
energia_key = normalizar_texto("Cobertura de energía eléctrica rural")
internet_key = normalizar_texto("Cobertura de Internet Rural (Censo)")

# --- A. PROCESAR COBERTURA DE ENERGÍA ELÉCTRICA RURAL (2021-2024) ---
df_vivienda_filtrado = df_vivienda[df_vivienda['indicador_clean'] == energia_key].copy()
df_vivienda_filtrado['departamento'] = df_vivienda_filtrado['departamento'].apply(normalizar_texto)

df_vivienda_filtrado['ano'] = pd.to_numeric(df_vivienda_filtrado['ano'], errors='coerce')
df_vivienda_filtrado = df_vivienda_filtrado[
    (df_vivienda_filtrado['ano'] >= 2021) & (df_vivienda_filtrado['ano'] <= 2024)
].copy()

df_vivienda_filtrado['dato_numerico'] = (
    df_vivienda_filtrado['dato_numerico']
    .astype(str)
    .str.replace(',', '.', regex=False)
    .str.strip()
)
df_vivienda_filtrado['dato_numerico'] = pd.to_numeric(df_vivienda_filtrado['dato_numerico'], errors='coerce')

df_energia_pivot = df_vivienda_filtrado.pivot_table(
    index=['departamento', 'entidad', 'ano'],
    values='dato_numerico',
    aggfunc='mean'
).reset_index()

df_energia_dept = df_energia_pivot.groupby(['departamento', 'ano'])['dato_numerico'].mean().reset_index()
df_energia_dept.columns = ['departamento', 'ano', 'cobertura_energia_electrica_rural']

# --- B. EXTRAER INTERNET RURAL DE 2018 HISTÓRICO ---
df_internet_raw = df_vivienda[df_vivienda['indicador_clean'] == internet_key].copy()
df_internet_raw['dato_numerico_clean'] = (
    df_internet_raw['dato_numerico']
    .astype(str)
    .str.replace(',', '.', regex=False)
    .str.strip()
)
df_internet_raw['dato_numerico_clean'] = pd.to_numeric(df_internet_raw['dato_numerico_clean'], errors='coerce')
df_internet_raw['ano'] = pd.to_numeric(df_internet_raw['ano'], errors='coerce')

# Filtramos por el año de censo (2018) y normalizamos departamentos
df_internet_2018 = df_internet_raw[df_internet_raw['ano'] == 2018].copy()
df_internet_2018['departamento'] = df_internet_2018['departamento'].apply(normalizar_texto)

df_internet_pivot_2018 = df_internet_2018.pivot_table(
    index=['departamento', 'entidad'],
    values='dato_numerico_clean',
    aggfunc='mean'
).reset_index()

df_internet_dept_2018 = df_internet_pivot_2018.groupby('departamento')['dato_numerico_clean'].mean().reset_index()
df_internet_dept_2018.columns = ['departamento', 'cobertura_internet_rural_censo']

# --- C. CREAR LA ESTRUCTURA DE AÑOS 2021-2024 Y REPLICAR EL DATO DE INTERNET ---
departamentos_unicos = df_energia_dept['departamento'].unique()
anios_interes = [2021, 2022, 2023, 2024]

estructura_anios = pd.DataFrame([
    {'departamento': dep, 'ano': anio}
    for dep in departamentos_unicos
    for anio in anios_interes
])

df_internet_replicado = pd.merge(estructura_anios, df_internet_dept_2018, on='departamento', how='left')

# --- D. UNIFICAR AMBOS INDICADORES DE VIVIENDA ---
df_vivienda_consolidado = pd.merge(
    df_energia_dept,
    df_internet_replicado,
    on=['departamento', 'ano'],
    how='outer'
)

# Sincronizamos con el nombre original esperado del DataFrame de salida para el notebook
df_vivienda_departamental = df_vivienda_consolidado.copy()

print("Datos de Vivienda Rural filtrados, promediados por Departamento (Internet 2018 propagado):")
display(df_vivienda_departamental.head(15))
print(f"\nFilas obtenidas: {len(df_vivienda_departamental)}")
print(df_vivienda_departamental.info())

Datos de Vivienda Rural filtrados, promediados por Departamento (Internet 2018 propagado):


,departamento,ano,cobertura_energia_electrica_rural,cobertura_internet_rural_censo
0,amazonas,2021,35.516667,1.990000
1,amazonas,2022,36.490000,1.990000
2,amazonas,2023,61.606667,1.990000
3,amazonas,2024,36.110000,1.990000
4,antioquia,2021,96.984524,8.656960
5,antioquia,2022,97.446270,8.656960
6,antioquia,2023,93.841270,8.656960
7,antioquia,2024,93.474921,8.656960
8,arauca,2021,78.451429,2.250000
9,arauca,2022,91.310000,2.250000



Filas obtenidas: 136
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 136 entries, 0 to 135
Data columns (total 4 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   departamento                       136 non-null    object 
 1   ano                                136 non-null    int64  
 2   cobertura_energia_electrica_rural  136 non-null    float64
 3   cobertura_internet_rural_censo     132 non-null    float64
dtypes: float64(2), int64(1), object(1)
memory usage: 4.4+ KB
None


### **Integración de Nivel de Educación (Analfabetismo) - Encuesta de Calidad de Vida (ECV)**

Descargamos la base de datos `anex-ECV-Series-2024.xlsx` y procesamos la hoja **Cuadro 17** para obtener el total de personas sin ningún nivel de educación por departamento para los años 2021 a 2024.

In [ ]:
import pandas as pd
import requests

# URL del archivo excel en github
excel_url_ecv = "https://github.com/emily-escudero/Analitica-Educacion-rural/raw/main/anex-ECV-Series-2024.xlsx"
ecv_filename = "anex-ECV-Series-2024.xlsx"

# Descargar el archivo excel
print(f"Downloading {excel_url_ecv}...")
response = requests.get(excel_url_ecv)
response.raise_for_status()

with open(ecv_filename, 'wb') as f:
    f.write(response.content)

print(f"Archivo descargado exitosamente: {ecv_filename}")

Archivo descargado exitosamente: anex-ECV-Series-2024.xlsx


In [ ]:
import pandas as pd
import numpy as np

# 1. Cargar la hoja 'Cuadro 17' completa
df_raw = pd.read_excel(ecv_filename, sheet_name='Cuadro 17', header=None)

# 2. Definir las columnas de interés para la categoría 'Ninguno' de cada año
col_2021 = 30  # Columna AE (Ninguno - 2021)
col_2022 = 39  # Columna AN (Ninguno - 2022)
col_2023 = 48  # Columna AW (Ninguno - 2023)
col_2024 = 57  # Columna BF (Ninguno - 2024)

registros_directos = []
departamento_actual = None

# Recorrer fila por fila el Excel
for idx, fila in df_raw.iterrows():
    col_0 = fila[0]
    col_1 = fila[1]

    # Si la celda de la columna 0 no está vacía, actualizamos el departamento actual
    if pd.notna(col_0):
        val_text = str(col_0).strip()
        # Ignoramos metadatos del encabezado y filas que no son departamentos individuales
        if not val_text.startswith(('1', '2', '3', 'Actualizado', '*', 'Fuente', 'Nota', 'Dominio', 'Cabecera', 'Centros')):
            if 'nacional' not in val_text.lower() and 'colombia' not in val_text.lower():
                departamento_actual = val_text

    # Si tenemos un departamento identificado y la fila corresponde al rango de edad 'Total'
    if departamento_actual is not None and pd.notna(col_1) and str(col_1).strip().lower() == 'total':
        p_2021 = pd.to_numeric(fila[col_2021], errors='coerce')
        p_2022 = pd.to_numeric(fila[col_2022], errors='coerce')
        p_2023 = pd.to_numeric(fila[col_2023], errors='coerce')
        p_2024 = pd.to_numeric(fila[col_2024], errors='coerce')

        registros_directos.append({
            'departamento': departamento_actual,
            'nivel_educativo': 'Ninguno',
            'porcentaje_2021': p_2021,
            'porcentaje_2022': p_2022,
            'porcentaje_2023': p_2023,
            'porcentaje_2024': p_2024
        })

# Crear el DataFrame de salida
df_analfabetismo_consolidado = pd.DataFrame(registros_directos)

print(f"DataFrame generado exitosamente con {len(df_analfabetismo_consolidado)} registros:")
display(df_analfabetismo_consolidado)


DataFrame generado exitosamente con 33 registros:


,departamento,nivel_educativo,porcentaje_2021,porcentaje_2022,porcentaje_2023,porcentaje_2024
0,Amazonas,Ninguno,3.289689,3.206196,3.338666,2.744884
1,Antioquia,Ninguno,3.641840,3.073455,3.289673,3.284072
2,Arauca,Ninguno,4.206663,4.333972,5.174291,4.007139
3,Atlántico,Ninguno,3.259863,2.954476,2.835493,3.546208
4,Bogotá D.C.,Ninguno,1.166187,1.139023,1.064514,0.881018
5,Bolívar,Ninguno,5.733140,5.232853,5.114274,4.563562
6,Boyacá,Ninguno,3.211914,3.069255,2.714362,2.581040
7,Caldas,Ninguno,3.066858,3.293731,3.254603,2.909432
8,Caquetá,Ninguno,4.611308,5.055848,4.639663,3.263663
9,Casanare,Ninguno,3.526339,2.859278,3.469665,3.259150


### **Consolidación de Todas las Bases de Datos (2021-2024)**

En este paso unificamos toda la información recolectada en una base de datos maestra donde la llave primaria es la combinación de `departamento` y `ano`. Para evitar problemas de unión, normalizamos los nombres de los departamentos y transformamos el DataFrame de analfabetismo ECV a formato largo.

In [ ]:
import re
import unicodedata
import pandas as pd

ANOS = range(2021, 2025)
LLAVE = ['departamento_norm', 'ano']

def limpiar_departamento(texto):
    """Nombre normalizado para cruzar fuentes: minúsculas, sin tildes ni signos."""
    if not isinstance(texto, str):
        return ""
    t = unicodedata.normalize('NFKD', texto.lower()).encode('ascii', 'ignore').decode()
    t = re.sub(r'[^a-z0-9\s]', '', t)
    t = re.sub(r'\s+', ' ', t).strip()
    for clave, canonico in [('bogota', 'bogota dc'), ('valle', 'valle'), ('san andres', 'san andres')]:
        if clave in t:
            return canonico
    return t

def preparar(df, conservar_nombre=False):
    """Agrega la llave normalizada, fija 'ano' como entero y excluye la fila 'Colombia'."""
    out = df.assign(departamento_norm=df['departamento'].map(limpiar_departamento),
                    ano=df['ano'].astype(int))
    out = out[out['departamento_norm'] != 'colombia']
    return out if conservar_nombre else out.drop(columns='departamento')

# --- 1. Preparar cada fuente (base: deserción) ---
base = preparar(df_deserciones_2021_2024, conservar_nombre=True)

analfabetismo = (
    df_analfabetismo_consolidado
    .melt(id_vars=['departamento', 'nivel_educativo'],
          value_vars=[f'porcentaje_{a}' for a in ANOS],
          var_name='ano', value_name='porcentaje_analfabetismo_ecv')
    .assign(ano=lambda d: d['ano'].str.extract(r'(\d+)', expand=False).astype(int))
    .drop(columns='nivel_educativo')   # si hay varios niveles, filtra uno ANTES de este paso
)

# Sedes: un solo año (EDUC 2024), se repite en los 4 años
sedes = df_instituciones.merge(pd.DataFrame({'ano': list(ANOS)}), how='cross')

fuentes = {
    'pobreza': preparar(df_pobreza_completos),
    'vivienda': preparar(df_vivienda_departamental),
    'analfabetismo': preparar(analfabetismo),
    'sedes': preparar(sedes),
}

# --- 2. Cruce: left desde deserción + validaciones ---
filas_base = len(base)
for nombre, tabla in fuentes.items():
    assert not tabla.duplicated(LLAVE).any(), f"'{nombre}' tiene la llave (departamento, año) duplicada"
    sin_match = set(tabla['departamento_norm']) - set(base['departamento_norm'])
    if sin_match:
        print(f"Aviso: en '{nombre}' hay departamentos sin match en deserción: {sorted(sin_match)}")
    base = base.merge(tabla, on=LLAVE, how='left')
assert len(base) == filas_base, "El cruce cambió el número de filas"

# --- 3. Formato final ---
base.loc[base['departamento_norm'] == 'bogota dc', 'departamento'] = 'Bogotá D.C.'
COLUMNAS = [
    'departamento', 'ano',
    'desercion', 'desercion_transicion', 'desercion_primaria', 'desercion_secundaria', 'desercion_media',
    'pobreza_monetaria', 'pobreza_extrema',
    'cobertura_energia_electrica_rural', 'cobertura_internet_rural_censo',
    'porcentaje_analfabetismo_ecv',
    'total_sedes_urbanas', 'total_sedes_rurales', 'total_sedes',
]
maestro_df = base[COLUMNAS].sort_values(['departamento', 'ano']).reset_index(drop=True)   # KeyError si falta alguna

print(f"Base maestra: {maestro_df.shape[0]} filas x {maestro_df.shape[1]} columnas")
print("Departamentos:", maestro_df['departamento'].nunique())
print("\nNulos por columna:")
print(maestro_df.isna().sum()[lambda s: s > 0])
display(maestro_df.head(8))

Base maestra: 132 filas x 15 columnas
Departamentos: 33

Nulos por columna:
pobreza_monetaria                 36
pobreza_extrema                   36
cobertura_internet_rural_censo     4
dtype: int64


,departamento,ano,desercion,desercion_transicion,desercion_primaria,desercion_secundaria,desercion_media,pobreza_monetaria,pobreza_extrema,cobertura_energia_electrica_rural,cobertura_internet_rural_censo,porcentaje_analfabetismo_ecv,total_sedes_urbanas,total_sedes_rurales,total_sedes
0,Amazonas,2021,5.04,6.93,3.31,6.51,7.38,NaN,NaN,35.516667,1.99000,3.289689,47,161,208
1,Amazonas,2022,6.11,6.08,4.04,8.90,7.78,NaN,NaN,36.490000,1.99000,3.206196,47,161,208
2,Amazonas,2023,5.01,5.31,3.26,6.77,7.79,NaN,NaN,61.606667,1.99000,3.338666,47,161,208
3,Amazonas,2024,5.11,6.95,2.95,7.74,6.2,NaN,NaN,36.110000,1.99000,2.744884,47,161,208
4,Antioquia,2021,4.81,3.59,4.27,5.90,4.13,32.8,9.2,96.984524,8.65696,3.641840,4126,9853,13979
5,Antioquia,2022,4.73,4.36,4.01,6.00,3.82,28.8,9.8,97.446270,8.65696,3.073455,4126,9853,13979
6,Antioquia,2023,4.83,4.05,4.09,6.33,3.75,27.5,8.8,93.841270,8.65696,3.289673,4126,9853,13979
7,Antioquia,2024,3.78,3.35,3.07,5.1,2.94,24.7,7.8,93.474921,8.65696,3.284072,4126,9853,13979


## Duplicados y datos faltantes


In [ ]:
import pandas as pd
import numpy as np

LLAVE = ['departamento', 'ano']
COLS_PORCENTAJE = [
    'pobreza_monetaria', 'pobreza_extrema', 'cobertura_energia_electrica_rural',
    'cobertura_internet_rural_censo', 'porcentaje_analfabetismo_ecv'
]

def resumen_nulos(df):
    """Tabla con las columnas que tienen nulos (cantidad y porcentaje)."""
    nulos = df.isna().sum()
    nulos = nulos[nulos > 0].sort_values(ascending=False)
    return pd.DataFrame({
        'Nulos': nulos,
        'Porcentaje (%)': (nulos / len(df) * 100).round(2)
    })

def tratar_duplicados(df):
    """Cuenta y elimina duplicados por (departamento, año) sin modificar el original."""
    df = df.copy()
    n_antes = len(df)
    dup_completos = df.duplicated().sum()
    dup_llave = df.duplicated(subset=LLAVE).sum()

    print("=== DUPLICADOS ===")
    print(f"Filas antes: {n_antes} | Repetidas completas: {dup_completos} | Repetidas por llave: {dup_llave}")

    if dup_llave > 0:
        print("\nFilas duplicadas por llave:")
        display(df[df.duplicated(subset=LLAVE, keep=False)].sort_values(LLAVE))
        df = df.drop_duplicates(subset=LLAVE, keep='first')
        print(f"Filas después: {len(df)}")

    assert not df.duplicated(subset=LLAVE).any(), "Quedaron duplicados por departamento-año."
    return df

def tratar_faltantes(df):
    """Revisa nulos y ceros, y corrige el 0 de cobertura eléctrica de San Andrés (2023-2024)."""
    df = df.copy()

    print("\n=== FALTANTES (ANTES) ===")
    display(resumen_nulos(df))

    # Nulos en pobreza_monetaria
    nulos_pobreza = df.loc[df['pobreza_monetaria'].isna(), LLAVE].sort_values(LLAVE)
    if not nulos_pobreza.empty:
        print("Departamentos con nulos en 'pobreza_monetaria':")
        display(nulos_pobreza)

    # Ceros en columnas de porcentaje (solo conteo por columna)
    ceros = (df[COLS_PORCENTAJE] == 0).sum()
    ceros = ceros[ceros > 0]
    if not ceros.empty:
        print("Ceros por columna de porcentaje (para revisión):")
        display(ceros.to_frame('Ceros'))

    # Corregir el cero de San Andrés en cobertura eléctrica rural
    filtro = (
        df['departamento'].str.lower().str.contains('andres|providencia')
        & df['ano'].isin([2023, 2024])
        & (df['cobertura_energia_electrica_rural'] == 0)
    )
    df.loc[filtro, 'cobertura_energia_electrica_rural'] = np.nan
    print(f"Ceros reemplazados por NaN (San Andrés y Providencia, 2023-2024): {filtro.sum()}")

    print("\n=== FALTANTES (DESPUÉS) ===")
    display(resumen_nulos(df))

    return df

# --- Ejecución ---
df_sin_duplicados = tratar_duplicados(maestro_df)
maestro_df_limpio = tratar_faltantes(df_sin_duplicados)

=== DUPLICADOS ===
Filas antes: 132 | Repetidas completas: 0 | Repetidas por llave: 0

=== FALTANTES (ANTES) ===


,Nulos,Porcentaje (%)
pobreza_monetaria,36,27.27
pobreza_extrema,36,27.27
cobertura_internet_rural_censo,4,3.03


Departamentos con nulos en 'pobreza_monetaria':


,departamento,ano
0,Amazonas,2021
1,Amazonas,2022
2,Amazonas,2023
3,Amazonas,2024
8,Arauca,2021
9,Arauca,2022
10,Arauca,2023
11,Arauca,2024
12,"Archipiélago de San Andrés, Providencia y Sant...",2021
13,"Archipiélago de San Andrés, Providencia y Sant...",2022


Ceros reemplazados por NaN (San Andrés y Providencia, 2023-2024): 0

=== FALTANTES (DESPUÉS) ===


,Nulos,Porcentaje (%)
pobreza_monetaria,36,27.27
pobreza_extrema,36,27.27
cobertura_internet_rural_censo,4,3.03


El diagnóstico identificó datos faltantes en las variables de pobreza monetaria, pobreza extrema y cobertura de internet rural, principalmente por limitaciones estadísticas y geográficas. Estos valores se conservaron como nulos para evitar distorsiones en el análisis. Además, dos valores de 0 % en la cobertura de energía eléctrica rural se reemplazaron por nulos, al considerarse posibles errores de registro.

## Categorización

In [ ]:
import pandas as pd
import numpy as np

# 1. Tratamiento especial para San Andrés y Providencia (reemplazar ceros de 2023 y 2024 por NaN)
filtro_san_andres = (
    (maestro_df['departamento'].str.lower().str.contains('andres|providencia')) &
    (maestro_df['ano'].isin([2023, 2024])) &
    (maestro_df['cobertura_energia_electrica_rural'] == 0)
)
maestro_df.loc[filtro_san_andres, 'cobertura_energia_electrica_rural'] = np.nan

# 2. Definir cortes, etiquetas y categorizar con pd.cut (right=False)
bins = [0, 50, 80, 95, 100.01]
etiquetas = ['muy baja', 'baja', 'media', 'alta']

# Generar la categorización
categoria_series = pd.cut(
    maestro_df['cobertura_energia_electrica_rural'],
    bins=bins,
    labels=etiquetas,
    right=False
)

# 3. Convertir a tipo categórico ordenado de forma explícita
maestro_df['nivel_cobertura_electrica'] = pd.Categorical(
    categoria_series,
    categories=etiquetas,
    ordered=True
)

# 4. Mostrar el conteo por categoría (incluyendo valores nulos)
print("Conteo por categoría (incluyendo NaN):")
print(maestro_df['nivel_cobertura_electrica'].value_counts(dropna=False))
print("\n" + "="*50 + "\n")

# 5. Mostrar la tabla de contingencia de departamentos por categoría y año
tabla_contingencia = pd.crosstab(
    index=maestro_df['nivel_cobertura_electrica'],
    columns=maestro_df['ano'],
    dropna=False
)
print("Número de departamentos por categoría y año:")
display(tabla_contingencia)

Conteo por categoría (incluyendo NaN):
nivel_cobertura_electrica
baja        57
media       46
muy baja    21
alta         8
Name: count, dtype: int64


Número de departamentos por categoría y año:


ano,2021,2022,2023,2024
nivel_cobertura_electrica,,,,
muy baja,7,4,5,5
baja,10,14,17,16
media,13,11,11,11
alta,3,4,0,1


### Justificación de la Categorización de Cobertura Eléctrica

Esta clasificación permite conservar los datos originales para los análisis numéricos y, al mismo tiempo, agruparlos en niveles que facilitan su interpretación. Se utilizaron rangos fijos porque permiten identificar mejor las diferencias en la cobertura eléctrica; por ejemplo, una cobertura menor al 50 % indica una situación crítica. Además, los valores de cero de San Andrés y Providencia en 2023 y 2024 se cambiaron por datos nulos, ya que podrían deberse a errores en los registros y no a la ausencia total del servicio.

In [ ]:
import pandas as pd

# 1. Convertir 'desercion' a número y revisar nulos
nulos_antes = maestro_df['desercion'].isna().sum()
maestro_df['desercion'] = pd.to_numeric(maestro_df['desercion'], errors='coerce')
nulos_despues = maestro_df['desercion'].isna().sum()
print(f"Nulos en 'desercion' - Antes: {nulos_antes} | Después: {nulos_despues}\n")

# 2. Nivel de deserción por terciles dentro de cada año
def terciles(serie):
    # rank(method='first') evita problemas con empates; los nulos se mantienen como NaN
    return pd.qcut(serie.rank(method='first'), q=3, labels=['baja', 'media', 'alta'])

maestro_df['nivel_desercion'] = (
    maestro_df.groupby('ano')['desercion']
    .transform(terciles)
    .astype(pd.CategoricalDtype(['baja', 'media', 'alta'], ordered=True))
)

# 3. Bandera: ¿está por encima del promedio de su año?
promedio_anual = maestro_df.groupby('ano')['desercion'].transform('mean')
maestro_df['sobre_promedio_nac'] = (maestro_df['desercion'] > promedio_anual).astype(object)
maestro_df.loc[maestro_df['desercion'].isna(), 'sobre_promedio_nac'] = None

# (a) Departamentos en cada nivel por año
print("=== (a) Departamentos por nivel de deserción y año ===")
display(pd.crosstab(maestro_df['nivel_desercion'], maestro_df['ano'], dropna=False))

# (b) Departamentos sobre el promedio nacional por año
print("\n=== (b) Departamentos sobre el promedio nacional por año ===")
display(pd.crosstab(maestro_df['sobre_promedio_nac'].fillna('NaN'), maestro_df['ano']))

# (c) Departamentos en nivel 'alta' los 4 años
anos_en_alta = (
    maestro_df[maestro_df['nivel_desercion'] == 'alta']
    .groupby('departamento')['ano'].nunique()
)
siempre_alta = anos_en_alta[anos_en_alta == 4].index.tolist()

print("\n=== (c) Departamentos con nivel 'alta' en los 4 años (2021-2024) ===")
if siempre_alta:
    print("\n".join(f"- {dep}" for dep in siempre_alta))
else:
    print("Ningún departamento se mantuvo en nivel 'alta' durante los 4 años.")

Nulos en 'desercion' - Antes: 0 | Después: 0

=== (a) Departamentos por nivel de deserción y año ===


ano,2021,2022,2023,2024
nivel_desercion,,,,
baja,11,11,11,11
media,11,11,11,11
alta,11,11,11,11



=== (b) Departamentos sobre el promedio nacional por año ===


/tmp/ipykernel_3432/130825030.py:31: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  display(pd.crosstab(maestro_df['sobre_promedio_nac'].fillna('NaN'), maestro_df['ano']))


ano,2021,2022,2023,2024
sobre_promedio_nac,,,,
False,19,18,17,17
True,14,15,16,16



=== (c) Departamentos con nivel 'alta' en los 4 años (2021-2024) ===
- Amazonas
- Caquetá
- Putumayo
- Quindio
- Risaralda
- Vichada


### Justificación de la Clasificación de Deserción Escolar

Esta metodología mantiene los valores numéricos originales para conservar la precisión de los datos y los complementa con categorías que facilitan su interpretación. Los niveles de deserción se calculan por separado para cada año, permitiendo comparar los territorios según las condiciones de cada periodo. Además, la variable sobre_promedio_nac permite identificar fácilmente qué territorios tienen una deserción superior al promedio nacional.